# 03 Benchmark Result Summary

**Scope.** Summarize benchmark metrics across molecular representations and models, while refusing to invent absent performance files or derived values.

## Expected input files

- `results/benchmark_metrics/macro_auc_by_model_representation.csv`
- `results/benchmark_metrics/micro_auc_by_model_representation.csv`
- `results/benchmark_metrics/labelwise_delta_auc.csv`
- `results/structure_disjoint/structure_disjoint_metrics.csv`

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import yaml
import matplotlib.pyplot as plt

SAVE_OUTPUTS = False


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        has_configs = (candidate / "configs").is_dir()
        has_readme = (candidate / "README.md").exists()
        if has_configs and has_readme:
            return candidate
        if has_configs:
            fallback = candidate
            return fallback
    raise FileNotFoundError("Could not locate project root containing configs/.")


def load_yaml(path: Path):
    with path.open("r", encoding="utf-8") as fh:
        return yaml.safe_load(fh)


def check_files(paths):
    rows = []
    for path in paths:
        path = Path(path)
        rows.append({"path": path.as_posix(), "exists": path.exists()})
    return pd.DataFrame(rows)


def require_files(paths):
    missing = [Path(path) for path in paths if not Path(path).exists()]
    if missing:
        formatted = "
".join(f"- {path.as_posix()}" for path in missing)
        raise FileNotFoundError(
            "The following required files are missing:
"
            f"{formatted}
"
            "Data required from user later before this notebook can run end-to-end."
        )


def print_missing_file_report(report_df: pd.DataFrame) -> None:
    missing = report_df.loc[~report_df["exists"], "path"].tolist()
    if missing:
        print("Data required from user later:")
        for item in missing:
            print(f"- {item}")
    else:
        print("All expected input files are present.")


PROJECT_ROOT = find_project_root()
CONFIG_DIR = PROJECT_ROOT / "configs"
DATASET_CFG = load_yaml(CONFIG_DIR / "dataset.yaml")


## 1. Scope

This notebook summarizes released benchmark metrics and representation comparisons when result files are available. It does not infer or fabricate performance values in the absence of result tables.

## 2. Expected inputs

The notebook expects processed benchmark metric summaries for standard and structure-disjoint evaluation.

## 3. Load configs

In [ ]:
split_cfg = load_yaml(CONFIG_DIR / 'split_settings.yaml')
model_cfg = load_yaml(CONFIG_DIR / 'model_hyperparameters.yaml')
expected_files = [
    PROJECT_ROOT / 'results/benchmark_metrics/macro_auc_by_model_representation.csv',
    PROJECT_ROOT / 'results/benchmark_metrics/micro_auc_by_model_representation.csv',
    PROJECT_ROOT / 'results/benchmark_metrics/labelwise_delta_auc.csv',
    PROJECT_ROOT / 'results/structure_disjoint/structure_disjoint_metrics.csv',
]
print('Benchmark seeds:', split_cfg['random_seeds'])
print('Configured benchmark models:', [k for k in model_cfg.keys() if k != 'metrics'])

## 4. File existence checks

In [ ]:
file_report = check_files(expected_files)
display(file_report)
print_missing_file_report(file_report)

## 5. Load macro-AUC and micro-AUC summaries

In [ ]:
loaded = {}
name_map = {
    'macro_auc': expected_files[0],
    'micro_auc': expected_files[1],
    'labelwise_delta_auc': expected_files[2],
    'structure_disjoint': expected_files[3],
}
for name, path in name_map.items():
    if path.exists():
        loaded[name] = pd.read_csv(path)
        print(f'Loaded {name}:', loaded[name].shape)
    else:
        print(f'Missing {name}: {path.as_posix()}')

## 6. Compare representations across six models

In [ ]:
if {'macro_auc', 'micro_auc'} <= set(loaded):
    for metric_name in ['macro_auc', 'micro_auc']:
        df = loaded[metric_name]
        display(df.head())
        required_cols = {'model', 'representation'}
        if required_cols.issubset(df.columns):
            print(metric_name, 'unique models:', sorted(df['model'].astype(str).unique().tolist()))
            print(metric_name, 'unique representations:', sorted(df['representation'].astype(str).unique().tolist()))
        else:
            print(metric_name, 'is missing one or more expected identifier columns.')
else:
    print('Representation comparison requires both macro-AUC and micro-AUC summary tables.')

## 7. Calculate or verify remaining error reduction if required columns exist

In [ ]:
if 'macro_auc' in loaded:
    df = loaded['macro_auc'].copy()
    required_cols = {'representation', 'model', 'macro_auc'}
    if required_cols.issubset(df.columns):
        best_by_model = df.groupby('model', as_index=False)['macro_auc'].max().rename(columns={'macro_auc': 'best_macro_auc'})
        display(best_by_model)
        if {'baseline_macro_auc', 'improved_macro_auc'}.issubset(df.columns):
            denom = 1 - df['baseline_macro_auc']
            valid = denom != 0
            df.loc[valid, 'remaining_error_reduction'] = (df.loc[valid, 'improved_macro_auc'] - df.loc[valid, 'baseline_macro_auc']) / denom.loc[valid]
            display(df[['model', 'representation', 'remaining_error_reduction']].head())
        else:
            print('Remaining error reduction not computed because baseline/improved columns are absent.')
    else:
        print('Macro-AUC table does not contain the expected columns for verification.')
else:
    print('Remaining error reduction checks require the macro-AUC summary table.')

## 8. Summarize label-wise ?AUC

In [ ]:
if 'labelwise_delta_auc' in loaded:
    delta_df = loaded['labelwise_delta_auc']
    display(delta_df.head())
    if 'delta_auc' in delta_df.columns:
        summary = delta_df['delta_auc'].describe()
        display(summary.to_frame(name='delta_auc_summary'))
    else:
        print('Column `delta_auc` not found; please adapt this template to the released schema if needed.')
else:
    print('Label-wise ?AUC summary requires labelwise_delta_auc.csv.')

## 9. Summarize structure-disjoint split performance

In [ ]:
if 'structure_disjoint' in loaded:
    disjoint_df = loaded['structure_disjoint']
    display(disjoint_df.head())
    print('Rows in structure-disjoint summary:', len(disjoint_df))
else:
    print('Structure-disjoint summary requires results/structure_disjoint/structure_disjoint_metrics.csv.')

## 10. Missing result checklist

In [ ]:
missing_items = file_report.loc[~file_report['exists'], 'path'].tolist()
if missing_items:
    print('Data required from user later:')
    for item in missing_items:
        print(f'- {item}')
else:
    print('No missing benchmark result files detected for this notebook.')